# AION Chat — Transformer on GPU (Colab)

**Phase 2 of 2: Fine-tune the ~235M Transformer on chat data using pretrained weights.**

This is the **GPU** counterpart to `colab_chat_tpu.ipynb`. It fine-tunes the base you
pretrained (e.g. `aion-transformer-tpu-large`) on a single T4, or on 2x T4 via DDP.

**Target: 20,000 steps across sessions (~4 sessions x 5,000 steps).** Each session
checkpoints to Drive; re-run cells 1-6 then 7 to resume.

## Before first run
1. `Runtime -> Change runtime type -> T4 GPU` (a **GPU**, not TPU).
2. Add Colab secrets (sidebar -> Key icon): `GITHUB_TOKEN`, `GITHUB_USERNAME`.
3. Upload the pretrained base folder to Drive at
   `MyDrive/aion_checkpoints/aion-transformer-tpu-large/` (must contain `best.pt` or
   `latest.pt` **and** `tokenizer.json`). The local name `transformer_tpu_large` also works.

The base is TPU-trained but loads fine on GPU (checkpoints are device-portable).

In [ ]:
# Cell 1 - Install dependencies, clone repo, verify GPU
import subprocess, sys, os, gc
from google.colab import userdata

TOKEN           = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')

if not GITHUB_USERNAME:
    raise ValueError("Add a Colab secret named 'GITHUB_USERNAME' (sidebar -> Key icon).")
if not TOKEN:
    raise ValueError("Add a Colab secret named 'GITHUB_TOKEN' (sidebar -> Key icon).")

REPO_URL = f'https://{TOKEN}@github.com/{GITHUB_USERNAME}/aion.git'

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets', 'tokenizers', 'pyyaml', 'tqdm',
], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)

if not os.path.exists('/content/aion'):
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, '/content/aion'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/aion', 'pull', '--ff-only'], check=True)
    print('Repo up to date.')

if '/content/aion/src' not in sys.path:
    sys.path.insert(0, '/content/aion/src')

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

# Verify a supported GPU is attached (needs sm_70+; T4 is sm_75).
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Set Runtime -> Change runtime type -> T4 GPU.')
_cc = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)} (sm_{_cc[0]}{_cc[1]}, '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB), count={torch.cuda.device_count()}')
if _cc[0] < 7:
    raise RuntimeError(f'GPU sm_{_cc[0]}{_cc[1]} is unsupported (needs sm_70+). Use a T4 runtime.')
print('Done.')

In [ ]:
# Cell 2 - Mount Drive + check existing progress
from google.colab import drive
from pathlib import Path
import json

drive.mount('/content/drive')

# Only 'large' (~235M) is wired here. The chat checkpoints live in their own Drive folder.
MODEL_SIZE = 'large'
_CHAT_DIR = {'large': 'transformer_gpu_chat_large'}[MODEL_SIZE]

CKPT_DIR = Path('/content/drive/MyDrive/aion_checkpoints') / _CHAT_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

latest = CKPT_DIR / 'latest.pt'
if latest.exists():
    import torch
    meta = torch.load(latest, map_location='cpu', weights_only=False)
    steps_done = meta.get('step', 0)
    TARGET_STEPS = 20000
    print(f'Progress: {steps_done:,} / {TARGET_STEPS:,} steps ({steps_done / TARGET_STEPS * 100:.1f}%)')
    if meta.get('metrics_log'):
        vals = [e for e in meta['metrics_log'] if 'val_loss' in e]
        if vals:
            print(f'Last val_loss: {vals[-1]["val_loss"]:.4f}')
    del meta
else:
    print('No chat checkpoint found - this is the first session.')

print(f'Model size: {MODEL_SIZE} | Checkpoint dir: {CKPT_DIR}')

In [ ]:
# Cell 3 - Download chat datasets
import sys, runpy
from pathlib import Path

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

RAW_DIR = '/content/data/raw'
sys.argv = ['cli', 'download', '--target', RAW_DIR, '--preset', 'chat']
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
print('Download complete.')

In [ ]:
# Cell 4 - Merge datasets + tokenizer
# The chat model MUST use the same tokenizer as the pretrained base (same vocab).
import shutil, sys, runpy, json
from pathlib import Path

RAW_DIR    = Path('/content/data/raw')
DATA_DIR   = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the pretrain base folder on Drive (local name or Kaggle-download name).
_DRIVE_CKPTS = Path('/content/drive/MyDrive/aion_checkpoints')
_BASE_CANDIDATES = {
    'large': ['transformer_tpu_large', 'aion-transformer-tpu-large'],
}[globals().get('MODEL_SIZE', 'large')]
PRETRAIN_CKPT_DIR = next((_DRIVE_CKPTS / n for n in _BASE_CANDIDATES if (_DRIVE_CKPTS / n).exists()),
                         _DRIVE_CKPTS / _BASE_CANDIDATES[0])

SEED_PATH      = '/content/aion/src/llm_lab/data/instruction_seed.json'
MERGED_PATH    = '/content/data/chat_merged.json'
TOKENIZER_PATH = '/content/data/tokenizer.json'
PRETRAIN_TOK   = PRETRAIN_CKPT_DIR / 'tokenizer.json'
DRIVE_TOK      = CKPT_DIR / 'tokenizer.json'

# Reuse tokenizer: prefer the pretrain base (must match the model weights' vocab).
if PRETRAIN_TOK.exists():
    shutil.copy2(PRETRAIN_TOK, TOKENIZER_PATH)
    print('Tokenizer restored from pretrain base.')
    need_tokenizer = False
elif DRIVE_TOK.exists():
    shutil.copy2(DRIVE_TOK, TOKENIZER_PATH)
    print('Tokenizer restored from Drive.')
    need_tokenizer = False
else:
    need_tokenizer = True

# Always rebuild chat_merged.json (ephemeral disk).
sys.argv = [
    'cli', 'merge-chat',
    '--raw-dir', str(RAW_DIR),
    '--out',     MERGED_PATH,
    '--seed',    SEED_PATH,
    '--max-hh-rlhf', '20000',
]
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)

if need_tokenizer:
    chat_examples = json.loads(Path(MERGED_PATH).read_text())
    corpus_path = DATA_DIR / 'corpus.txt'
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for ex in chat_examples:
            for msg in ex['messages']:
                f.write(msg['content'].strip() + '\n')
    sys.argv = ['cli', 'tokenizer', '--corpus', str(corpus_path), '--out', TOKENIZER_PATH, '--vocab-size', '16384']
    runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
    shutil.copy2(TOKENIZER_PATH, DRIVE_TOK)
    print('Tokenizer trained and saved to Drive.')

print('Data ready.')

In [ ]:
# Cell 5 - Drive logger
import sys, time, threading, json
from datetime import datetime
from pathlib import Path
from tqdm import tqdm as _tqdm

LOG_PATH    = CKPT_DIR / 'training.log'
STATUS_PATH = CKPT_DIR / 'status.txt'

class _Tee:
    def __init__(self, original, log_path):
        self._orig = original
        self._log  = open(log_path, 'a', encoding='utf-8', buffering=1)
    def write(self, data):
        self._orig.write(data)
        self._log.write(data)
    def flush(self):
        self._orig.flush()
        self._log.flush()
    def __getattr__(self, attr):
        return getattr(self._orig, attr)

sys.stdout = _Tee(sys.stdout, LOG_PATH)
print(f'\n=== Session started {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} ===')

def _monitor():
    metrics_path = CKPT_DIR / 'metrics.json'
    while True:
        time.sleep(60)
        try:
            if metrics_path.exists():
                data = json.loads(metrics_path.read_text())
                train = [e for e in data if 'train_loss' in e]
                val   = [e for e in data if 'val_loss' in e]
                lines = [
                    f'Updated:    {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
                    f'Step:       {train[-1]["step"] if train else "n/a"}',
                    f'Train loss: {train[-1]["train_loss"]:.4f}' if train else 'Train loss: n/a',
                    f'Val loss:   {val[-1]["val_loss"]:.4f} (step {val[-1]["step"]})' if val else 'Val loss: n/a',
                    f'Best val:   {min(e["val_loss"] for e in val):.4f} @ step {min(val, key=lambda x: x["val_loss"])["step"]}' if val else 'Best val: n/a',
                ]
                STATUS_PATH.write_text('\n'.join(lines) + '\n')
                _tqdm.write(f'[monitor] {lines[0]} | {lines[1]} | {lines[2]}')
        except Exception:
            pass

threading.Thread(target=_monitor, daemon=True).start()
print(f'Drive logger active - log: {LOG_PATH.name}, status: {STATUS_PATH.name}')

In [ ]:
# Cell 6 - Train / resume on GPU
import sys, runpy, yaml, shutil, os, gc, json
import torch
from pathlib import Path

STEPS_PER_SESSION = 5000
# True = both T4s via DDP (~1.6x faster); False = single T4 (safe fallback).
# Effective batch is kept at 32 either way (2 GPUs x batch 2 x accum 8 = 32).
USE_BOTH_GPUS = True

_MODEL_SIZE = globals().get('MODEL_SIZE', 'large')

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

gc.collect()
torch.cuda.empty_cache()

# If pretrained weights exist but no chat checkpoint yet, seed from the base (fresh optimizer).
_DRIVE_CKPTS = Path('/content/drive/MyDrive/aion_checkpoints')
_BASE_CANDIDATES = {
    'large': ['transformer_tpu_large', 'aion-transformer-tpu-large'],
}[_MODEL_SIZE]
PRETRAIN_CKPT_DIR = next((_DRIVE_CKPTS / n for n in _BASE_CANDIDATES if (_DRIVE_CKPTS / n).exists()),
                         _DRIVE_CKPTS / _BASE_CANDIDATES[0])
latest = CKPT_DIR / 'latest.pt'

if not latest.exists():
    pretrain_best = PRETRAIN_CKPT_DIR / 'best.pt'
    pretrain_latest = PRETRAIN_CKPT_DIR / 'latest.pt'
    pretrain_src = pretrain_best if pretrain_best.exists() else pretrain_latest
    if pretrain_src.exists():
        _ckpt = torch.load(pretrain_src, map_location='cpu', weights_only=False)
        _ckpt['step'] = 0
        _ckpt['optimizer'] = None
        _ckpt['scheduler'] = None
        _ckpt['metrics_log'] = []
        torch.save(_ckpt, latest)
        del _ckpt
        gc.collect()
        print(f'Seeded chat training from pretrained base: {pretrain_src.name}')
    else:
        raise FileNotFoundError(
            f'No base checkpoint in {PRETRAIN_CKPT_DIR}. Upload the pretrained folder to Drive '
            f'(needs best.pt or latest.pt + tokenizer.json).')

# Fresh seed has step 0 and empty metrics -> use the full-LR config; a real resume uses low-LR.
metrics_path = CKPT_DIR / 'metrics.json'
if metrics_path.exists():
    _m = json.loads(metrics_path.read_text())
    steps_done = max((e.get('step', 0) for e in _m), default=0)
else:
    steps_done = 0

_CFG = {
    'large': ('transformer_gpu_chat_large.yaml', 'transformer_gpu_chat_large_resume.yaml'),
}[_MODEL_SIZE]
_CFG_DIR = Path('/content/aion/src/llm_lab/configs')

if steps_done > 0:
    print(f'Found checkpoint at step {steps_done:,} - resuming with low LR.')
    cfg_path = _CFG_DIR / _CFG[1]
else:
    print('Fresh seed - starting with full LR.')
    cfg_path = _CFG_DIR / _CFG[0]

cfg = yaml.safe_load(cfg_path.read_text())
print(f'Using config: {cfg_path.name}')

if USE_BOTH_GPUS and torch.cuda.device_count() > 1:
    cfg['gpus'] = 0              # DistributedDataParallel across all visible GPUs
    cfg['grad_accum_steps'] = 8  # 2 GPUs x batch 2 x accum 8 = eff batch 32
else:
    cfg['gpus'] = 1

cfg['dataset_type']     = 'chat'
cfg['instruction_data'] = '/content/data/chat_merged.json'
cfg['train_path']       = ''
cfg['val_path']         = ''
cfg['tokenizer_path']   = '/content/data/tokenizer.json'
cfg['checkpoint_dir']   = str(CKPT_DIR)
cfg['max_steps']        = steps_done + STEPS_PER_SESSION

run_cfg_path = '/content/run_chat_gpu.yaml'
Path(run_cfg_path).write_text(yaml.dump(cfg))
shutil.copy2(run_cfg_path, CKPT_DIR / 'run.yaml')

_ngpu = torch.cuda.device_count() if cfg['gpus'] == 0 else 1
print(f'Steps this session: {STEPS_PER_SESSION} (step {steps_done:,} -> {cfg["max_steps"]:,})')
print(f'gpus={cfg["gpus"]} (0=all via DDP, 1=single), eff_batch={cfg["batch_size"]*cfg["grad_accum_steps"]*_ngpu}')
print(f'lr={cfg["lr"]:.1e}, warmup={cfg["warmup_steps"]}, seq_len={cfg["seq_len"]}, grad_checkpoint={cfg["grad_checkpoint"]}')
print(f'Model: d_model={cfg["d_model"]}, layers={cfg["n_layers"]}, heads={cfg["n_heads"]}')
print(f'Checkpoints -> {CKPT_DIR}')
print()

try:
    if '/content/aion/src' not in sys.path:
        sys.path.insert(0, '/content/aion/src')
    sys.argv = ['cli', 'train', '--config', run_cfg_path]
    runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
finally:
    for _key in list(sys.modules.keys()):
        if _key.startswith('llm_lab'):
            del sys.modules[_key]
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Cell 7 - Test the model
import sys, torch
from pathlib import Path
from tokenizers import Tokenizer

sys.path.insert(0, '/content/aion/src')
from llm_lab.training.config import TrainConfig
from llm_lab.training.model_factory import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = Tokenizer.from_file('/content/data/tokenizer.json')
cfg = TrainConfig.load(Path('/content/run_chat_gpu.yaml'))

model = build_model(cfg).to(device)

best_ckpt = CKPT_DIR / 'best.pt'
latest_ckpt = CKPT_DIR / 'latest.pt'
ckpt_path = best_ckpt if best_ckpt.exists() else latest_ckpt
if not ckpt_path.exists():
    raise FileNotFoundError(f'No checkpoint found in {CKPT_DIR}')

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()
print(f'Loaded {ckpt_path.name} from step {ckpt["step"]:,}')

def chat(user_message: str, max_new_tokens: int = 200, temperature: float = 0.8) -> str:
    prompt = f'<|user|>{user_message}<|end|>\n<|assistant|>'
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)
    end_id = tokenizer.encode('<|end|>').ids[0]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == end_id:
                break
    generated = tokenizer.decode(input_ids[0].tolist())
    marker = '<|assistant|>'
    if marker in generated:
        generated = generated.split(marker, 1)[1].replace('<|end|>', '').strip()
    return generated

print('User: What is machine learning?')
print('Assistant:', chat('What is machine learning?'))